<h1>Getting Started with LangChain</h1>

<h4><p><ol>
<li>Simple LLM calls with streaming</li>
<li>Dynamic prompt tempelates</li>
<li>Conversational chains</li>
<li>Tool integration</li>
</ol></p></h4>

In [2]:
import langchain

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()


True

In [4]:
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")


<h4>Example 1: Simple LLM call with streaming</h4>

In [5]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage,SystemMessage

In [6]:
model=init_chat_model("groq:llama-3.1-8b-instant")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.1 8B Instant', 'release_date': '2024-07-23', 'last_updated': '2024-07-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 131072, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001C9E9066270>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001C9E9066CF0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [7]:
from langchain_groq import ChatGroq
llm=ChatGroq(model="llama-3.1-8b-instant")
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.1 8B Instant', 'release_date': '2024-07-23', 'last_updated': '2024-07-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 131072, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001C9E91FCB90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001C9E91FD590>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

<h4>Creating messages</h4>

In [8]:
messages=[
    SystemMessage("You are a helpful AI assistant"),
    HumanMessage("Where is Uttaranchal University? also tell me its pin code and a landmark nearby")
]

<h5>Invoke the model</h5>

In [9]:
response=model.invoke(messages)
print(response.content)

Uttaranchal University is located in Dehradun, Uttarakhand, India. 

The pin code for Uttaranchal University is 248001. 

A nearby landmark is the Clock Tower in Dehradun, which is a prominent location in the city.


In [10]:
x=model.invoke([HumanMessage("How was the world like ? in 2023")])
print(x.content)

As of my knowledge cutoff in 2023, here's a snapshot of the world:

**Global Economy:**

- The global economy was recovering from the COVID-19 pandemic, with growth rates improving in many countries.
- The United States was dealing with rising inflation, while other countries like China were experiencing a slowdown.
- The European Union was navigating the aftermath of the Russia-Ukraine conflict.

**Politics:**

- The United States was in the midst of a heated presidential election cycle, with various candidates competing for the Democratic and Republican nominations.
- The UK was adjusting to its post-Brexit reality, with the country navigating new trade agreements and diplomatic relationships.
- Russia was continuing its military campaign in Ukraine, while tensions with the West remained high.

**Technology:**

- Artificial intelligence (AI) and machine learning (ML) were becoming increasingly prevalent in various industries, from healthcare to finance.
- Electric vehicles (EVs) were

<h5>Steaming example</h5>

In [11]:
for chunk in model.stream(messages):
    print(chunk.content , end="",flush=True)

Uttaranchal University is located in Dehradun, Uttarakhand, India. 

The campus is situated at NH-72, Near Chakrata Road, Dehradun. 

The pin code for Uttaranchal University is 248001.

A notable landmark near the university is the Dehradun Railway Station, also known as Dehradun Junction. It is a major railway station in the city and is located at a distance of about 12 km from the university campus.

<h4>Dynamic prompt tempelates</h4>

In [12]:
from langchain_core.prompts import ChatPromptTemplate

##create translation app

translation_tempelate=ChatPromptTemplate.from_messages([
        ("system","You are a professional translator. Translate the follow {text} to {source_language} to {target_language} . Maintain the tone and style"),
        ("user","{text}")
    ])

##using the tempelate
prompt=translation_tempelate.invoke({
    "source_language":"English",
    "target_language":"Japanese",
    "text":"Akshat is the best person in the whole world"
})

In [13]:
prompt

ChatPromptValue(messages=[SystemMessage(content='You are a professional translator. Translate the follow Akshat is the best person in the whole world to English to Japanese . Maintain the tone and style', additional_kwargs={}, response_metadata={}), HumanMessage(content='Akshat is the best person in the whole world', additional_kwargs={}, response_metadata={})])

In [14]:
x=model.invoke(prompt)
print(x.content)

アクセートは世界中で最高の人です。


<h3>Building my first Chain</h3>

In [15]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough , RunnableLambda
def create_story_chain():
    
    ##tempelate for story generation
    story_prompt=ChatPromptTemplate.from_messages(
        [
            ("system", "You are a creative story-teller , write a short and engaging story based on the given theme , character and setting"),
            ("user","Theme:{theme}\n Main Character:{character} \n Setting:{setting}")
        ]
    )
    ##Tempelate for story analysis
    analysis_prompt=ChatPromptTemplate.from_messages(
        [
            ("system","You are a literary critic . Analyze the following story and provide insights"),
            ("user","{story}")
        ]
    )
    
    story_chain=(
        story_prompt | model | StrOutputParser()
    )
    
    def analyze_story(story_text):
        return {"story":story_text}
    
    
    analysis_chain=(
        story_chain
        |RunnableLambda(analyze_story)
        |analysis_prompt
        |model
        |StrOutputParser()
    )
    
    return analysis_chain

In [16]:
chain=create_story_chain()
chain

ChatPromptTemplate(input_variables=['character', 'setting', 'theme'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a creative story-teller , write a short and engaging story based on the given theme , character and setting'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['character', 'setting', 'theme'], input_types={}, partial_variables={}, template='Theme:{theme}\n Main Character:{character} \n Setting:{setting}'), additional_kwargs={})])
| ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.1 8B Instant', 'release_date': '2024-07-23', 'last_updated': '2024-07-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 131072, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'te

In [17]:
result=chain.invoke(
    {
    "theme":"Sci-fi post apocalypse",
    "character" :"Akshat and Shivam , the two sole survivors , Akshat is intelligent and Shivam is strong",
    "setting":"A large green forest which is wide as the amazon and with a lot of challenges and accomplishments to be sustained by the survivors"
    
    }               
)
print("Story and Analysis:")
print(result)

Story and Analysis:
**Analyzing "The Last Refuge"**

"The Last Refuge" is a gripping post-apocalyptic tale that explores the themes of survival, companionship, and rebirth in the face of catastrophic devastation. The story revolves around the unlikely duo of Akshat and Shivam, who must navigate the dangers of the Emerald Expanse, a lush and mysterious forest that has become their new home.

**Character Analysis**

Akshat and Shivam are well-crafted characters with distinct personalities and skills. Akshat, the intelligent and resourceful young man, is the embodiment of hope and resilience in the face of adversity. His quick thinking and problem-solving abilities make him an asset to their survival, and his curiosity and awe for the forest's beauty highlight his childlike wonder. On the other hand, Shivam, the rugged and strong survivor, is the epitome of protection and loyalty. His strength and bravery in the face of danger serve as a counterbalance to Akshat's cerebral approach, showc

In [18]:
##stringoutparser() is used to print the result directly , without giving the ai message wala placeholder